In [ ]:
import pandas as pd
import numpy as np

#  1 LOAD 
RAW_FILE = "Student_raw.csv"

# Row 0 = short column names
# Row 1 = full question text
# Row 2 = ImportId junk
df_raw = pd.read_csv(RAW_FILE, header=0, skiprows=[1, 2], low_memory=False)

# Optional!?! keep full question text for reference
q_labels = pd.read_csv(RAW_FILE, header=None, nrows=3).iloc[1].to_dict()

# Rename the actual columns from Qualtrics 
position_rename = {
    " ": "consent",
    "Q1": "library_engagement",
    " .1": "frequent_use_reason",
    " .2": "low_use_reason",
    "Q2": "essential_service",
    " .3": "digital_resources_secret",
    " .4": "study_space_value",
    " .5": "book_librarian_impact",
    " .6": "other_service_text",
    "Q3": "health_info_confidence",
    " .7": "search_process_challenge",
    " .8": "health_info_advice",
    "Q4": "course_materials_satisfaction",
    " .9": "missing_course_textbook",
    " .10": "course_materials_explain",
    " .11": "course_materials_access_location",
    "Q5": "academic_success_story",
    "Q6": "exam_support_resource",
    " .12": "exam_support_stress_reduction",
    "Q7_NPS_GROUP": "library_nps_group",
    "Q7": "library_nps_score",
    " .13": "nps_reason",
    " .14": "name",
    " .15": "degree_program",
    "ResponseId": "response_id",
    "RecordedDate": "recorded_date",
    "Duration (in seconds)": "duration_seconds",
    "Progress": "progress_pct",
}

df = df_raw.rename(columns=position_rename).copy()


#  2 REMOVE TEST / INCOMPLETE RESPONSES 
if "Status" in df.columns:
    df = df[df["Status"] != "Survey Preview"].copy()

if "Finished" in df.columns:
    df["Finished"] = df["Finished"].astype(str).str.upper().str.strip()
    df = df[df["Finished"].isin(["TRUE", "1"])].copy()

df = df.reset_index(drop=True)


#  3 DROP USELESS / SENSITIVE COLs 
df = df.dropna(axis=1, how="all")

drop_cols = [
    "IPAddress", "RecipientLastName", "RecipientFirstName",
    "RecipientEmail", "ExternalReference",
    "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage",
    "Q_RecaptchaScore", "Q_DuplicateRespondent"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])


#  4. FiX COLs TYPES 
for col in ["StartDate", "EndDate", "recorded_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "duration_seconds" in df.columns:
    df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")

if "progress_pct" in df.columns:
    df["progress_pct"] = pd.to_numeric(df["progress_pct"], errors="coerce")

# fields were exported as labels, so map them to numbers
health_info_map = {
    "1 Not confident": 1,
    "Not confident": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5 Extremely confident": 5,
    "Extremely confident": 5
}

course_materials_map = {
    "1 Very dissatisfied": 1,
    "Very dissatisfied": 1,
    "2 Dissatisfied": 2,
    "Dissatisfied": 2,
    "3 Neutral": 3,
    "Neutral": 3,
    "4 Satisfied": 4,
    "Satisfied": 4,
    "5 Very satisfied": 5,
    "Very satisfied": 5
}

if "health_info_confidence" in df.columns:
    df["health_info_confidence_num"] = df["health_info_confidence"].map(health_info_map)

if "course_materials_satisfaction" in df.columns:
    df["course_materials_satisfaction_num"] = df["course_materials_satisfaction"].map(course_materials_map)

if "library_nps_score" in df.columns:
    df["library_nps_score"] = pd.to_numeric(df["library_nps_score"], errors="coerce")
# Check for straggler values
if "health_info_confidence_num" in df.columns:
    unmapped_health = df[
        df["health_info_confidence"].notna() & df["health_info_confidence_num"].isna()
    ]["health_info_confidence"].unique()
    if len(unmapped_health) > 0:
        print(f" Unmapped health confidence values: {unmapped_health}")

if "course_materials_satisfaction_num" in df.columns:
    unmapped_course = df[
        df["course_materials_satisfaction"].notna() & df["course_materials_satisfaction_num"].isna()
    ]["course_materials_satisfaction"].unique()
    if len(unmapped_course) > 0:
        print(f"Unmapped course satisfaction values: {unmapped_course}")
        
if "library_nps_score" in df.columns:
    def nps_bucket(score):
        if pd.isna(score):
            return np.nan
        elif score >= 9:
            return "Promoter"
        elif score >= 7:
            return "Passive"
        else:
            return "Detractor"

    df["library_nps_group_calc"] = df["library_nps_score"].apply(nps_bucket)

if "duration_seconds" in df.columns:
    df["duration_minutes"] = (df["duration_seconds"] / 60).round(1)

if "recorded_date" in df.columns:
    df["survey_month"] = df["recorded_date"].dt.to_period("M").astype(str)

# Standardize degree program labels just in case casing varies later
if "degree_program" in df.columns:
    df["degree_program"] = df["degree_program"].astype(str).str.strip().str.upper()
    
#  6 split into QUANT + TxT 
quant_cols = [
    "response_id", "recorded_date", "survey_month",
    "duration_minutes", "progress_pct",
    "health_info_confidence_num",
    "course_materials_satisfaction_num",
    "library_nps_score",
    "library_nps_group",
    "library_nps_group_calc",
    "degree_program"
]
quant_cols = [c for c in quant_cols if c in df.columns]
df_quant = df[quant_cols].copy()

text_cols = [
    "response_id",
    "frequent_use_reason", "low_use_reason",
    "digital_resources_secret", "study_space_value",
    "book_librarian_impact", "other_service_text",
    "search_process_challenge", "health_info_advice",
    "missing_course_textbook", "course_materials_explain",
    "course_materials_access_location",
    "academic_success_story",
    "exam_support_resource", "exam_support_stress_reduction",
    "nps_reason"
]
text_cols = [c for c in text_cols if c in df.columns]
df_text = df[text_cols].copy()


# 7 export
df.to_csv("student_clean_full.csv", index=False)
df_quant.to_csv("student_quant.csv", index=False)
df_text.to_csv("student_text.csv", index=False)

print(f"✓ Cleaned rows: {len(df)}")
print(f"✓ Quant columns: {len(df_quant.columns)}")
print(f"✓ Text columns: {len(df_text.columns)}")

if len(df) == 0:
    print("WARNING: No rows survived filtering. Check Status and Finished.")
else:
    print("\nSample stats for numeric fields:")
    rating_cols = [
        c for c in [
            "health_info_confidence_num",
            "course_materials_satisfaction_num",
            "library_nps_score"
        ] if c in df.columns
    ]
    if rating_cols:
        print(df[rating_cols].describe().round(2))




✓ Cleaned rows: 8
✓ Quant columns: 11
✓ Text columns: 11

Sample stats for numeric fields:
       health_info_confidence_num  course_materials_satisfaction_num  \
count                        6.00                               6.00   
mean                         3.00                               3.67   
std                          1.26                               0.82   
min                          1.00                               3.00   
25%                          2.25                               3.00   
50%                          3.50                               3.50   
75%                          4.00                               4.00   
max                          4.00                               5.00   

       library_nps_score  
count               6.00  
mean                7.50  
std                 2.07  
min                 5.00  
25%                 5.75  
50%                 8.00  
75%                 8.75  
max                10.00  
['2' '1 Not confi

In [19]:
import pandas as pd

faculty_text = pd.read_csv("faculty_text.csv")
student_text = pd.read_csv("student_text.csv")

print("=== FACULTY: library value to leadership ===")
print(faculty_text["library_value_to_leadership"].dropna().tolist())

print("\n=== STUDENT: nps reason ===")
print(student_text["nps_reason"].dropna().tolist())

print("\n=== STUDENT: health info advice ===")
print(student_text["health_info_advice"].dropna().tolist())

=== FACULTY: library value to leadership ===
['aasfdsfsdfds', 'Be more visible alongside the Office of Scholarly Activity. ', 'Just keep on keeping on', "I get consistent messaging that the Library is overextended and underresourced, so I hesitate to reach out to the Library. For example, it would be useful to have a Librarian come to one of my courses and describe best practices for searching databases. \n\nI think there's also an issue with the Library scope. Jan has communicated several big ideas over the past few years that would really stretch the current understanding at PNWU of what a Library is or should be. I can't tell if these are good ideas or not and neither can others, so we are kind of at an impasse of what to expect. "]

=== STUDENT: nps reason ===
['adfdafd', 'The library staff are always so helpful!!!', 'They are available, kind, and willing to help. ', 'Our library staff willingness to support our individual success!', 'the library team! ']

=== STUDENT: health info 